# Test 04: R + Scala/Java Combined Installation

Integration test for the most common multi-language combination.

**What this tests:**
1. Combined install (shared micromamba env, JDK used by Scala)
2. Package deduplication (no duplicate conda packages)
3. Both `%%R` and `%%scala` magics work side-by-side
4. Cross-language data flow: Python -> R -> Python -> Scala -> Python
5. Both languages can query Snowflake independently

**Prerequisites:**
- EAI with conda-forge, CRAN, Maven Central, GitHub, PyPI access

## 1. Install R + Scala Together

In [ ]:
# Install from local source (notebooks/ is inside the package dir)
!pip install -q ..

In [ ]:
from sfnb_multilang import install

report = install(languages=["r", "scala"])

In [ ]:
print(f"Success: {report.success}")
print(f"Env prefix: {report.env_prefix}")
print(f"Duration: {report.elapsed_seconds:.1f}s")
print()

for lang in ["r", "scala"]:
    pr = report.plugin_results.get(lang)
    vr = report.validation_results.get(lang)
    if pr:
        status = "PASS" if (pr.success and vr and vr.success) else "FAIL"
        print(f"  {lang}: {status} (version: {pr.version})")
        if pr.errors:
            print(f"    Errors: {pr.errors}")

## 2. Verify Shared Environment

Both R and Java should be in the same conda environment.

In [ ]:
import os

prefix = report.env_prefix
r_bin = os.path.join(prefix, "bin", "R")
java_bin = os.path.join(prefix, "bin", "java")

print(f"Env prefix: {prefix}")
print(f"R binary:    {r_bin}  exists={os.path.isfile(r_bin)}")
print(f"Java binary: {java_bin}  exists={os.path.isfile(java_bin)}")

assert os.path.isfile(r_bin), "FAIL: R binary not found in shared env"
assert os.path.isfile(java_bin), "FAIL: Java binary not found in shared env"
print("\nPASS: Both R and Java in same environment")

## 3. Set Up Both Magics

In [ ]:
from r_helpers import setup_r_environment
setup_r_environment()

In [ ]:
from scala_helpers import setup_scala_environment
setup_scala_environment()

## 4. R Smoke Test

In [ ]:
%%R
library(dplyr)
mtcars %>% group_by(cyl) %>% summarise(n = n(), avg_mpg = mean(mpg))
cat("PASS: R works\n")

## 5. Scala Smoke Test

In [ ]:
%%scala
val nums = (1 to 10).toList
println(s"Sum: ${nums.sum}, Mean: ${nums.sum.toDouble / nums.size}")
println("PASS: Scala works")

## 6. Cross-Language Data Pipeline

Python -> R (transform) -> Python -> Scala (process) -> Python

In [ ]:
import pandas as pd

raw_data = pd.DataFrame({
    "product": ["Widget", "Gadget", "Widget", "Gadget", "Widget"],
    "revenue": [100, 200, 150, 250, 175]
})
print("Step 1 - Raw data (Python):")
print(raw_data)

In [ ]:
%%R -i raw_data -o r_summary
# Step 2: Aggregate in R
library(dplyr)
r_summary <- raw_data %>%
  group_by(product) %>%
  summarise(total_revenue = sum(revenue), transactions = n())
r_summary

In [ ]:
# Step 3: Back in Python, inspect R result and extract scalar for Scala
print("Step 3 - R result in Python:")
print(r_summary)
assert len(r_summary) == 2, "FAIL: expected 2 products"
total_rev = float(r_summary["total_revenue"].sum())
num_products = len(r_summary)
print(f"Extracted: {num_products} products, total revenue = {total_rev}")

In [ ]:
%%scala -i total_rev,num_products
// Step 4: Receive scalar values from Python (R -> Python -> Scala)
val rev = total_rev.asInstanceOf[Double]
val prods = num_products.asInstanceOf[Int]
println(s"Step 4 - Received from R via Python: $prods products, total revenue = $rev")
println("PASS: Cross-language pipeline complete (R -> Python -> Scala)")

## 7. Both Query Snowflake

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
sf_info = session.sql("SELECT CURRENT_ACCOUNT() AS acct, CURRENT_WAREHOUSE() AS wh").to_pandas()
print(sf_info)

In [ ]:
%%R -i sf_info
cat("R sees Snowflake account:", sf_info$acct, "\n")
cat("PASS: R Snowflake connectivity\n")

In [ ]:
from scala_helpers import bootstrap_snowpark_scala
ok, msg = bootstrap_snowpark_scala(session)
print(f"Scala Snowpark bootstrap: {'OK' if ok else 'FAIL'}")
if msg:
    print(msg)

In [ ]:
%%scala
val acct = sfSession.sql("SELECT CURRENT_ACCOUNT()").collect()(0).getString(0)
println(s"Scala sees Snowflake account: $acct")
println("PASS: Scala Snowflake connectivity")

## Test Summary

| Test | Expected |
|---|---|
| Combined install | Both R and Scala in report.plugin_results |
| Shared env | R + Java binaries coexist in same prefix |
| R magic | `%%R` executes, dplyr works |
| Scala magic | `%%scala` executes, Scala code runs |
| Cross-language | Python -> R -> Python -> Scala pipeline |
| R -> Snowflake | R reads account info via Python proxy |
| Scala -> Snowflake | Scala connects via Snowpark session |